# Mini Model V2 - Correct Dataset Structure
**Based on actual competition data:**
- Training: Instrument stems (drums, vocals, bass, others)
- Test: Noisy mashups (mixed stems + ESC-50 noise)

**Strategy:**
1. Mix stems to simulate mashups during training
2. Add ESC-50 noise for robustness
3. Use research-backed settings (15s, 128 mels, SpecAugment)

In [34]:
!pip install -q librosa timm

import os
import glob
import random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [35]:
# Config
CONFIG = {
    'sr': 22050,
    'duration': 10,           # Reduced from 15 for speed
    'n_mels': 128,
    'n_fft': 2048,
    'hop_length': 512,
    'num_classes': 10,
    'batch_size': 24,         # Reduced for GPU memory
    'epochs': 5,
    'lr': 1e-3,
    'train_samples': 1000,    # Start small to validate
    'noise_prob': 0.7,
    'noise_level': (0.05, 0.3),
    'mixup_alpha': 0.3,
    'label_smoothing': 0.1,
}

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 
          'jazz', 'metal', 'pop', 'reggae', 'rock']
genre_to_idx = {g: i for i, g in enumerate(GENRES)}
STEMS = ['drums', 'vocals', 'bass', 'other']  # NOTE: 'other' not 'others'

print(f"Config: {CONFIG['train_samples']} samples, {CONFIG['epochs']} epochs, batch={CONFIG['batch_size']}")

Config: 1000 samples, 5 epochs, batch=24


In [36]:
# Paths - CORRECT for this competition
BASE_PATH = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR = os.path.join(BASE_PATH, 'genres_stems')
NOISE_DIR = os.path.join(BASE_PATH, 'ESC-50-master', 'audio')
TEST_DIR = os.path.join(BASE_PATH, 'mashups')
TEST_CSV = os.path.join(BASE_PATH, 'test.csv')
SAMPLE_SUB = os.path.join(BASE_PATH, 'sample_submission.csv')

print(f"Stems dir: {STEMS_DIR}")
print(f"Noise dir: {NOISE_DIR}")
print(f"Test dir: {TEST_DIR}")

# Verify paths exist
for p in [STEMS_DIR, TEST_DIR]:
    print(f"{p} exists: {os.path.exists(p)}")

Stems dir: /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems
Noise dir: /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio
Test dir: /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems exists: True
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups exists: True


In [37]:
# Explore data structure
print("=" * 50)
print("DATA STRUCTURE")
print("=" * 50)

if os.path.exists(STEMS_DIR):
    print(f"\nGenres in stems dir:")
    for genre in sorted(os.listdir(STEMS_DIR)):
        genre_path = os.path.join(STEMS_DIR, genre)
        if os.path.isdir(genre_path):
            songs = os.listdir(genre_path)
            print(f"  {genre}: {len(songs)} songs")
            # Show first song's stems
            if songs:
                first_song = os.path.join(genre_path, songs[0])
                if os.path.isdir(first_song):
                    stems = os.listdir(first_song)
                    print(f"    Sample stems: {stems[:4]}")

if os.path.exists(TEST_DIR):
    test_files = os.listdir(TEST_DIR)
    print(f"\nTest mashups: {len(test_files)} files")
    print(f"  Sample: {test_files[:3]}")

if os.path.exists(NOISE_DIR):
    noise_files = glob.glob(os.path.join(NOISE_DIR, '*.wav'))
    print(f"\nNoise files (ESC-50): {len(noise_files)}")

DATA STRUCTURE

Genres in stems dir:
  blues: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  classical: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  country: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  disco: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  hiphop: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  jazz: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  metal: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  pop: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  reggae: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  rock: 100 songs
    Sample stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']

Test mashups: 3020 files
  Sample: ['song2501.wav', 'song0

In [38]:
# Load noise files for augmentation
noise_files = glob.glob(os.path.join(NOISE_DIR, '*.wav')) if os.path.exists(NOISE_DIR) else []
print(f"Loaded {len(noise_files)} noise files for augmentation")

def load_noise(sr=22050, duration=15):
    """Load a random noise clip"""
    if not noise_files:
        return np.zeros(sr * duration)
    
    noise_path = random.choice(noise_files)
    try:
        noise, _ = librosa.load(noise_path, sr=sr, duration=duration)
        target_len = sr * duration
        if len(noise) < target_len:
            # Loop noise
            repeats = int(np.ceil(target_len / len(noise)))
            noise = np.tile(noise, repeats)[:target_len]
        else:
            noise = noise[:target_len]
        return noise
    except:
        return np.zeros(sr * duration)

Loaded 2000 noise files for augmentation


In [39]:
# Build training data index
# Structure: genres_stems/genre/song_id/{drums,vocals,bass,others}.wav

train_data = []  # List of (genre, song_path) tuples

for genre in GENRES:
    genre_dir = os.path.join(STEMS_DIR, genre)
    if os.path.exists(genre_dir):
        for song_id in os.listdir(genre_dir):
            song_path = os.path.join(genre_dir, song_id)
            if os.path.isdir(song_path):
                # Verify stems exist
                stems_exist = all(
                    os.path.exists(os.path.join(song_path, f"{stem}.wav"))
                    for stem in STEMS
                )
                if stems_exist:
                    train_data.append((genre, song_path))

print(f"Total songs with all stems: {len(train_data)}")

# Count per genre
genre_counts = {}
for genre, _ in train_data:
    genre_counts[genre] = genre_counts.get(genre, 0) + 1
print(f"Per genre: {genre_counts}")

Total songs with all stems: 1000
Per genre: {'blues': 100, 'classical': 100, 'country': 100, 'disco': 100, 'hiphop': 100, 'jazz': 100, 'metal': 100, 'pop': 100, 'reggae': 100, 'rock': 100}


In [40]:
# Audio loading and preprocessing
def load_and_mix_stems(song_path, sr=22050, duration=15):
    """Load all stems and mix them (simulating mashup creation)"""
    target_len = sr * duration
    mixed = np.zeros(target_len, dtype=np.float32)
    
    for stem in STEMS:
        stem_path = os.path.join(song_path, f"{stem}.wav")
        try:
            audio, _ = librosa.load(stem_path, sr=sr, duration=duration)
            if len(audio) < target_len:
                audio = np.pad(audio, (0, target_len - len(audio)))
            else:
                audio = audio[:target_len]
            mixed += audio
        except:
            pass
    
    # Normalize mixed audio
    if np.max(np.abs(mixed)) > 0:
        mixed = mixed / np.max(np.abs(mixed)) * 0.9
    
    return mixed.astype(np.float32)

def add_noise(audio, noise_level=0.1):
    """Add ESC-50 noise to audio"""
    noise = load_noise(CONFIG['sr'], CONFIG['duration'])
    # Normalize noise
    if np.max(np.abs(noise)) > 0:
        noise = noise / np.max(np.abs(noise))
    # Mix with specified level
    noisy_audio = audio + noise_level * noise
    # Renormalize
    if np.max(np.abs(noisy_audio)) > 0:
        noisy_audio = noisy_audio / np.max(np.abs(noisy_audio)) * 0.9
    return noisy_audio.astype(np.float32)

def preprocess(audio):
    """Basic preprocessing"""
    audio = audio - np.mean(audio)  # DC offset
    rms = np.sqrt(np.mean(audio**2)) + 1e-8
    audio = audio / rms * 0.1  # RMS normalize
    return np.clip(audio, -1, 1).astype(np.float32)

def audio_to_mel(audio, sr=22050):
    """Convert to mel spectrogram"""
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr,
        n_mels=CONFIG['n_mels'],
        n_fft=CONFIG['n_fft'],
        hop_length=CONFIG['hop_length']
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
    return mel_db

In [41]:
# SpecAugment
def spec_augment(mel, time_mask=20, freq_mask=10):
    mel = mel.copy()
    n_mels, n_frames = mel.shape
    
    # Time masking
    if n_frames > time_mask:
        t = np.random.randint(0, time_mask)
        t0 = np.random.randint(0, n_frames - t)
        mel[:, t0:t0+t] = 0
    
    # Frequency masking
    if n_mels > freq_mask:
        f = np.random.randint(0, freq_mask)
        f0 = np.random.randint(0, n_mels - f)
        mel[f0:f0+f, :] = 0
    
    return mel

In [42]:
# Dataset for STEMS
class StemDataset(Dataset):
    def __init__(self, data, augment=False):
        """
        data: list of (genre, song_path) tuples
        """
        self.data = data
        self.augment = augment
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        genre, song_path = self.data[idx]
        label = genre_to_idx[genre]
        
        # Load and mix stems
        audio = load_and_mix_stems(song_path, CONFIG['sr'], CONFIG['duration'])
        
        # Add noise (like test data)
        if self.augment and np.random.random() < CONFIG['noise_prob']:
            noise_level = np.random.uniform(*CONFIG['noise_level'])
            audio = add_noise(audio, noise_level)
        
        # Basic audio augmentation
        if self.augment:
            # Random gain
            if np.random.random() < 0.5:
                audio = audio * np.random.uniform(0.8, 1.2)
            # Time shift
            if np.random.random() < 0.5:
                shift = np.random.randint(-CONFIG['sr'], CONFIG['sr'])
                audio = np.roll(audio, shift)
        
        audio = preprocess(audio)
        mel = audio_to_mel(audio, CONFIG['sr'])
        
        # SpecAugment
        if self.augment:
            mel = spec_augment(mel)
        
        mel_tensor = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)
        return mel_tensor, label

# Dataset for TEST mashups
class MashupDataset(Dataset):
    def __init__(self, file_paths):
        self.file_paths = file_paths
        
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        path = self.file_paths[idx]
        try:
            audio, _ = librosa.load(path, sr=CONFIG['sr'], duration=CONFIG['duration'])
            target_len = CONFIG['sr'] * CONFIG['duration']
            if len(audio) < target_len:
                audio = np.pad(audio, (0, target_len - len(audio)))
            else:
                audio = audio[:target_len]
        except:
            audio = np.zeros(CONFIG['sr'] * CONFIG['duration'])
        
        audio = preprocess(audio)
        mel = audio_to_mel(audio, CONFIG['sr'])
        mel_tensor = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)
        return mel_tensor

In [43]:
# Prepare data
np.random.seed(42)
random.shuffle(train_data)

# Subsample if needed
n_samples = min(CONFIG['train_samples'], len(train_data))
train_data = train_data[:n_samples]

# Split
val_size = int(0.15 * len(train_data))
val_data = train_data[:val_size]
train_data_split = train_data[val_size:]

print(f"Train: {len(train_data_split)}, Val: {len(val_data)}")

train_loader = DataLoader(
    StemDataset(train_data_split, augment=True),
    batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2
)
val_loader = DataLoader(
    StemDataset(val_data, augment=False),
    batch_size=CONFIG['batch_size'], num_workers=2
)

Train: 850, Val: 150


In [44]:
# Model - Auto-detect feature dimensions
class GenreClassifier(nn.Module):
    def __init__(self, model_name='efficientnet_b0'):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        # Auto-detect feature size
        num_features = self.backbone.num_features
        print(f"Backbone features: {num_features}")
        
        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, CONFIG['num_classes'])
        )
    
    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

# Model options (uncomment one):
MODEL_NAME = 'efficientnet_b0'      # Fast, good accuracy
# MODEL_NAME = 'efficientnet_b2'    # Slower, better accuracy
# MODEL_NAME = 'resnet34'           # Classic, reliable

model = GenreClassifier(MODEL_NAME).to(device)
print(f"Model: {MODEL_NAME}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Backbone features: 1280
Model: efficientnet_b0
Total params: 4,338,054


In [45]:
# Mixup
def mixup_data(x, y, alpha=0.3):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [46]:
# Training
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=CONFIG['lr'], epochs=CONFIG['epochs'], steps_per_epoch=len(train_loader)
)

best_acc = 0
for epoch in range(CONFIG['epochs']):
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        
        # Mixup
        data, target_a, target_b, lam = mixup_data(data, target, CONFIG['mixup_alpha'])
        
        optimizer.zero_grad()
        output = model(data)
        loss = mixup_criterion(criterion, output, target_a, target_b, lam)
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        train_loss += loss.item()
        train_correct += (lam * (output.argmax(1) == target_a).float() + 
                         (1-lam) * (output.argmax(1) == target_b).float()).sum().item()
        train_total += target.size(0)
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    # Validate
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            val_correct += (output.argmax(1) == target).sum().item()
            val_total += target.size(0)
    
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
    
    print(f"Epoch {epoch+1}: Train={train_acc:.4f}, Val={val_acc:.4f}, Best={best_acc:.4f}")

print(f"\nBest Val Accuracy: {best_acc:.4f}")

Epoch 1: 100%|██████████| 36/36 [01:56<00:00,  3.24s/it, loss=1.7387]


Epoch 1: Train=0.2402, Val=0.4600, Best=0.4600


Epoch 2: 100%|██████████| 36/36 [01:28<00:00,  2.47s/it, loss=1.3474]


Epoch 2: Train=0.4870, Val=0.5800, Best=0.5800


Epoch 3: 100%|██████████| 36/36 [01:29<00:00,  2.48s/it, loss=2.2916]


Epoch 3: Train=0.5892, Val=0.7467, Best=0.7467


Epoch 4: 100%|██████████| 36/36 [01:29<00:00,  2.47s/it, loss=1.4570]


Epoch 4: Train=0.6750, Val=0.8200, Best=0.8200


Epoch 5: 100%|██████████| 36/36 [01:28<00:00,  2.46s/it, loss=1.3037]


Epoch 5: Train=0.7481, Val=0.8333, Best=0.8333

Best Val Accuracy: 0.8333


In [47]:
# Test inference
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

# Load test files from test.csv
if os.path.exists(TEST_CSV):
    test_df = pd.read_csv(TEST_CSV)
    print(f"Test CSV columns: {test_df.columns.tolist()}")
    print(f"Test samples: {len(test_df)}")
    
    # Get file paths
    if 'filename' in test_df.columns:
        test_files = [os.path.join(TEST_DIR, f) for f in test_df['filename']]
    elif 'id' in test_df.columns:
        # Try id as filename
        test_files = [os.path.join(TEST_DIR, f"{id}.wav") for id in test_df['id']]
    else:
        test_files = sorted(glob.glob(os.path.join(TEST_DIR, '*.wav')))
else:
    test_files = sorted(glob.glob(os.path.join(TEST_DIR, '*.wav')))

print(f"Test files: {len(test_files)}")
print(f"Sample: {test_files[:3]}")

Test CSV columns: ['id', 'filename']
Test samples: 3020
Test files: 3020
Sample: ['/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups/mashups/song0001.wav', '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups/mashups/song0002.wav', '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups/mashups/song0003.wav']


In [48]:
# Run inference
test_loader = DataLoader(MashupDataset(test_files), batch_size=CONFIG['batch_size'], num_workers=2)

predictions = []
with torch.no_grad():
    for data in tqdm(test_loader, desc="Testing"):
        data = data.to(device)
        output = model(data)
        predictions.extend(output.argmax(1).cpu().numpy())

print(f"Predictions: {len(predictions)}")

Testing: 100%|██████████| 126/126 [01:23<00:00,  1.51it/s]

Predictions: 3020


In [49]:
# Create submission
if os.path.exists(SAMPLE_SUB):
    sample_sub = pd.read_csv(SAMPLE_SUB)
    print(f"Sample submission columns: {sample_sub.columns.tolist()}")
    
    # Match format
    submission = sample_sub.copy()
    submission['genre'] = [GENRES[p] for p in predictions[:len(submission)]]
else:
    # Create from scratch
    submission = pd.DataFrame({
        'id': [os.path.basename(f).replace('.wav', '') for f in test_files],
        'genre': [GENRES[p] for p in predictions]
    })

submission.to_csv('submission.csv', index=False)
print("\nSaved submission.csv")
print(submission.head())
print(f"\nPrediction distribution:")
print(submission['genre'].value_counts())

Sample submission columns: ['id', 'genre']

Saved submission.csv
   id      genre
0   1  classical
1   2  classical
2   3  classical
3   4  classical
4   5  classical

Prediction distribution:
genre
classical    3020
Name: count, dtype: int64


In [50]:
# Assessment
print("\n" + "="*50)
print("MODEL ASSESSMENT")
print("="*50)
print(f"Best Val Accuracy: {best_acc:.4f}")
print(f"\nKey settings:")
print(f"  - Duration: {CONFIG['duration']}s")
print(f"  - Noise augmentation: {CONFIG['noise_prob']*100:.0f}% prob")
print(f"  - Noise level: {CONFIG['noise_level']}")
print(f"  - Mixup: {CONFIG['mixup_alpha']}")
print(f"  - SpecAugment: Yes")

if best_acc >= 0.75:
    print("\nVERDICT: Strong! Scale up for better results.")
elif best_acc >= 0.60:
    print("\nVERDICT: Decent. Consider more noise augmentation.")
else:
    print("\nVERDICT: Needs work. Try different approach.")


MODEL ASSESSMENT
Best Val Accuracy: 0.8333

Key settings:
  - Duration: 10s
  - Noise augmentation: 70% prob
  - Noise level: (0.05, 0.3)
  - Mixup: 0.3
  - SpecAugment: Yes

VERDICT: Strong! Scale up for better results.
